# 工具调用语法 Tool Use Syntax

**工具调用的本质**：
- LLM（大语言模型）本身不会直接调用工具。
- LLM 会"请求"开发者去调用某个工具，开发者负责执行该工具并将其结果返回给 LLM。
- 在开发社区中，人们常简化说"LLM 调用了工具"，但这在技术上并不准确，仅是一种便捷的说法。

- **智谱AI Python SDK**：
  - 这是智谱AI官方提供的 Python SDK (zai-sdk)。
  - 官方文档：https://docs.bigmodel.cn/cn/guide/models/free/glm-4.7-flash
  - 它提供了一种简便的语法来调用智谱AI的大模型（包括 GLM-4.7-Flash）。
  - 该库的核心功能之一是支持工具调用，极大简化了开发流程。

## 一、以 get_current_time 函数为例

In [1]:
import sys
sys.path.append('../')
from  utils import *

In [2]:
# 工具定义
from datetime import datetime

def get_current_time():
    """Returns current time as a string"""
    return datetime.now().strftime("%H:%M:%S")

In [2]:
# 安装智谱AI SDK（首次运行需要执行）
# pip install zai-sdk

In [6]:
# 使用智谱AI官方 Python SDK 调用 GLM-4.7-Flash
import os
# 创建客户端实例
# API Key 从https://open.bigmodel.cn/usercenter/apikeys 获取
client = zai_client
# 定义消息
messages = [
    {"role": "user", "content": "现在几点了？"}
]

# 定义工具描述（JSON Schema格式）
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns current time as a string",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

# 调用模型
response = client.chat.completions.create(
    model="glm-4.7-flash",     # 指定使用的模型
    messages=messages,          # 传递给 LLM 的消息数组
    tools=tools                 # 定义 LLM 可以访问的工具列表
)

# 处理工具调用
if (tool_call:=response.choices[0].message.tool_calls):
    # LLM请求调用工具
    tool_call = tool_call[0]
    tool_name = tool_call.function.name
    
    # 执行对应的工具函数
    if tool_name == "get_current_time":
        result = get_current_time()
        print_html (f"工具执行结果: {result}")
        
        # 将工具结果返回给LLM继续处理
        messages.append({
            "role": "assistant",
            "content": "",
            "tool_calls": [tool_call.model_dump()]
        })
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
        
        # 获取最终响应
        final_response = client.chat.completions.create(
            model="glm-4.7-flash",
            messages=messages
        )
        print_html(f"最终响应: {final_response.choices[0].message.content}")
else:
    # LLM直接响应，没有调用工具
    print(f"响应: {response.choices[0].message.content}")

### 参数解析：
- **model**: 选择要使用的 LLM，此处为智谱AI的 glm-4.7-flash。
- **messages**: 传递给模型的对话历史或提示信息。
- **tools**: 这是最核心的部分。需要以 JSON Schema 格式定义工具的描述，包括工具名称、描述和参数。
- **api_key**: 从 https://open.bigmodel.cn/usercenter/apikeys 获取的 API Key。

### 工具调用的处理流程：
1. 检查响应中是否包含 `tool_calls`
2. 如果有，提取工具名称和参数
3. 执行对应的工具函数
4. 将工具执行结果作为新的消息添加到对话历史中
5. 再次调用模型获取最终响应

## 二、工具描述的 JSON Schema 格式

智谱AI SDK 需要开发者手动以 JSON Schema 格式定义工具描述。这个 Schema 是传递给 LLM 的真实数据结构。

```json
{
  "type": "function",
  "function": {
    "name": "get_current_time",
    "description": "Returns current time as a string",
    "parameters": {
      "type": "object",
      "properties": {},
      "required": []
    }
  }
}
```

### JSON Schema 各字段说明：
- **type**: 固定为 "function"
- **function.name**: 工具函数的名称
- **function.description**: 工具函数的描述，帮助 LLM 理解工具用途
- **function.parameters**: 参数定义，包含类型和属性

## 三、智谱AI SDK 处理带参数的函数

In [7]:
# 带参数的工具定义
from datetime import datetime
from zoneinfo import ZoneInfo

def get_current_time(timezone):
    """Returns current time for given time zone"""
    timezone = ZoneInfo(timezone)
    return datetime.now(timezone).strftime("%H:%M:%S")

### 参数解析（带参数的工具）：

```json
{
  "type": "function",
  "function": {
    "name": "get_current_time",
    "description": "Returns current time for given time zone",
    "parameters": {
      "type": "object",
      "properties": {
        "timezone": {
          "type": "string",
          "description": "The IANA time zone string, e.g., 'America/New_York' or 'Pacific/Auckland'."
        }
      },
      "required": ["timezone"]
    }
  }
}
```

开发者需要手动定义 JSON Schema，包括参数的类型、描述以及哪些参数是必需的。这样，当 LLM 决定调用此函数时，它就知道需要提供一个类似 'America/New_York' 的字符串作为参数。

In [8]:
# 使用带参数的工具
# 示例：询问不同时区的时间
messages = [
    {"role": "user", "content": "纽约现在几点了？"}
]

# 定义带参数的工具
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Returns current time for given time zone",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "The IANA time zone string, e.g., 'America/New_York' or 'Pacific/Auckland'."
                    }
                },
                "required": ["timezone"]
            }
        }
    }
]

# 调用模型
response = client.chat.completions.create(
    model="glm-4.7-flash",
    messages=messages,
    tools=tools
)

# 处理工具调用
if (tool_call := response.choices[0].message.tool_calls):
    tool_call = tool_call[0]
    tool_name = tool_call.function.name
    
    # 解析参数
    import json
    tool_args = json.loads(tool_call.function.arguments)
    
    # 执行工具函数
    if tool_name == "get_current_time":
        result = get_current_time(tool_args.get("timezone"))
        print_html(f"工具执行结果: {result}")
        
        # 将工具结果返回给LLM
        messages.append({
            "role": "assistant",
            "content": "",
            "tool_calls": [tool_call.model_dump()]
        })
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result
        })
        
        # 获取最终响应
        final_response = client.chat.completions.create(
            model="glm-4.7-flash",
            messages=messages
        )
        print_html(f"最终响应: {final_response.choices[0].message.content}")

In [9]:
tool_call.function.arguments

'{"timezone":"America/New_York"}'

## 工作流程：

1. **开发者定义函数**：编写一个带有清晰 docstring 的 Python 函数。
2. **手动创建工具描述**：开发者需要手动以 JSON Schema 格式定义工具描述，包括名称、描述和参数。
3. **LLM 接收并决策**：LLM 接收到包含所有可用工具描述的 Schema。它会根据当前的对话上下文和用户需求，决定是否需要调用某个工具。
4. **LLM 发出请求**：如果需要，LLM 会生成一个包含工具名称和所需参数的请求。
5. **开发者执行工具**：开发者需要检查响应中的 `tool_calls`，提取参数后手动调用对应的函数。
6. **结果返回与迭代**：函数执行的结果需要作为新的消息添加到对话历史中，然后再次调用模型。需要手动管理这个过程。
7. **最终响应**：当模型不再调用工具时，返回最终的文本响应。

## 智谱AI Python SDK 的特点：

- **官方支持**：智谱AI官方提供的 Python SDK，充分测试和优化。
- **官方文档**：https://docs.bigmodel.cn/cn/guide/models/free/glm-4.7-flash
- **简单易用**：API 设计简洁，易于上手。
- **需要手动管理**：需要手动处理工具调用的完整流程，包括参数解析、函数执行和结果返回。
- **支持多种模型**：包括 glm-4.7-flash、glm-4 等多个智谱AI模型版本。
- **免费使用**：GLM-4.7-Flash 模型完全免费，适合学习和开发。

**快速开始**：
1. 安装SDK：`pip install zai-sdk`
2. 获取API Key：https://open.bigmodel.cn/usercenter/apikeys
3. 创建客户端：`client = ZhipuAiClient(api_key="your_api_key_here")`
4. 调用模型：`client.chat.completions.create(model="glm-4.7-flash", messages=messages)`